# Phase 2 — Feature sets for the three configurations (v3, revision 2)
Reads the Phase 1 split (outputs_v3/data) and writes ready-to-train arrays to outputs_v3/features_v2/<config>/.
* Config 2 (CICIDS2017) and Config 3 (CIC-IoT2023): all features except zero-variance columns (Wahab et al., 2025)
  and exact-duplicate columns (redundant features; Misrak & Melaku, 2025).
* Config 1 (merged): 13 unified features with the corrected mapping (iat_mean, total_size, header_length).
  syn_flag and cwe_flag are removed: in CICIDS2017, "SYN Flag Count" is identical to "Fwd PSH Flags" and
  "CWE Flag Count" to "Fwd URG Flags", so they do not measure what their names say.
* Non-negative wide-range features are transformed with log(1 + x) (Wahab et al., 2025), then Min-Max scaled
  (Wahab et al., 2025) with statistics from the training split only.

In [ ]:
# Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1 — settings and helpers
import os, json, time
import numpy as np
import pandas as pd

ROOT  = '/content/drive/MyDrive/Thesis/outputs_v3/'
DATA  = ROOT + 'data/'
FEAT  = ROOT + 'features_v2/'
SEED  = 42
CLASSES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
WIDE_RANGE = 1_000          # a feature whose training range (max - min) exceeds this is log-transformed
SPLITS = ['train', 'val', 'test']

def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)

def load(ds, split):
    return pd.read_parquet(DATA + f'{ds}_{split}.parquet')

def drop_redundant(train, feats):
    """Zero-variance columns and exact duplicate columns (first one kept), decided on the training split."""
    const = [c for c in feats if train[c].nunique(dropna=False) <= 1]
    keep, dup = [], {}
    for c in feats:
        if c in const:
            continue
        twin = next((k for k in keep if np.array_equal(train[c].to_numpy(), train[k].to_numpy())), None)
        if twin is None:
            keep.append(c)
        else:
            dup[c] = twin
    return keep, const, dup

def fit_transform(frames, feats):
    """log(1 + x) for non-negative wide-range features, then Min-Max with training statistics only."""
    tr = frames['train'][feats].to_numpy(np.float64)
    mn, mx = tr.min(0), tr.max(0)
    use_log = ((mx - mn) > WIDE_RANGE) & (mn >= 0)
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    out = {}
    for s, df in frames.items():
        x = df[feats].to_numpy(np.float64)
        x[:, use_log] = np.log1p(np.maximum(x[:, use_log], 0))   # these features are non-negative by definition
        out[s] = ((x - lo) / rng).astype(np.float32)
    params = dict(features=feats, train_min=mn.tolist(), train_max=mx.tolist(),
                  log_features=[f for f, u in zip(feats, use_log) if u], wide_range_threshold=WIDE_RANGE)
    return out, params

def save(cfg, X, frames, params, extra=None):
    d = FEAT + cfg + '/'
    os.makedirs(d, exist_ok=True)
    for s in SPLITS:
        np.save(d + f'X_{s}.npy', X[s])
        np.save(d + f'y_{s}.npy', frames[s]['label'].astype(str).map(CLASSES.index).to_numpy(np.int8))
        if 'source' in frames[s]:
            np.save(d + f'source_{s}.npy', frames[s]['source'].to_numpy().astype('U12'))
    json.dump(dict(params, classes=CLASSES, **(extra or {})), open(d + 'spec.json', 'w'), indent=2)
    log(f'{cfg}: saved {", ".join(f"{s} {X[s].shape}" for s in SPLITS)}')

In [ ]:
# Cell 2 — Config 2 (CICIDS2017) and Config 3 (CIC-IoT2023): native features
report = {}
for cfg, ds in [('config2_cicids2017', 'cicids2017'), ('config3_ciciot2023', 'ciciot2023')]:
    if os.path.exists(FEAT + cfg + '/spec.json'):
        log(f'{cfg}: already built, skipping'); continue
    frames = {s: load(ds, s) for s in SPLITS}
    feats_all = [c for c in frames['train'].columns if c not in ('label', 'label_orig')]
    feats, const, dup = drop_redundant(frames['train'], feats_all)
    X, params = fit_transform(frames, feats)
    save(cfg, X, frames, params, dict(dropped_zero_variance=const, dropped_duplicates=dup,
                                      n_features_original=len(feats_all)))
    report[cfg] = dict(original=len(feats_all), kept=len(feats), zero_variance=const, duplicates=dup,
                       log_features=len(params['log_features']))
    del frames, X

In [ ]:
# Cell 3 — Config 1 (merged): 13 unified features, corrected mapping
# syn_flag and cwe_flag removed: in CICIDS2017 'SYN Flag Count' == 'Fwd PSH Flags' and 'CWE Flag Count' == 'Fwd URG Flags'
SAME = {  # unified -> (CICIDS2017 column, CIC-IoT2023 column)
    'fin_flag': ('FIN Flag Count', 'fin_flag_number'),
    'rst_flag': ('RST Flag Count', 'rst_flag_number'), 'psh_flag': ('PSH Flag Count', 'psh_flag_number'),
    'ack_flag': ('ACK Flag Count', 'ack_flag_number'), 'ece_flag': ('ECE Flag Count', 'ece_flag_number'),
    'min_pkt_len': ('Min Packet Length', 'Min'), 'max_pkt_len': ('Max Packet Length', 'Max'),
    'mean_pkt_len': ('Packet Length Mean', 'AVG'), 'pkt_variance': ('Packet Length Variance', 'Variance'),
    'pkt_std': ('Packet Length Std', 'Std'),
}
UNIFIED = list(SAME) + ['iat_mean', 'total_size', 'header_length']

def unify_ids(df):
    u = pd.DataFrame({k: df[v[0]] for k, v in SAME.items()})
    u['iat_mean'] = df['Flow IAT Mean'].clip(lower=0)                                           # microseconds
    u['total_size'] = df['Total Length of Fwd Packets'] + df['Total Length of Bwd Packets']     # bytes in the flow
    u['header_length'] = (df['Fwd Header Length'].clip(lower=0) /
                          df['Total Fwd Packets'].clip(lower=1)).clip(upper=60)                  # bytes per packet
    u['label'], u['source'] = df['label'].astype(str).values, 'CICIDS2017'
    return u

def unify_iot(df):
    u = pd.DataFrame({k: df[v[1]] for k, v in SAME.items()})
    u['iat_mean'] = df['IAT'].clip(lower=0) * 1e6                                              # seconds -> microseconds
    u['total_size'] = df['Tot sum']                                                            # sum of packet lengths
    u['header_length'] = df['Header_Length']
    u['label'], u['source'] = df['label'].astype(str).values, 'CIC-IoT2023'
    return u

cfg = 'config1_merged'
if os.path.exists(FEAT + cfg + '/spec.json'):
    log(f'{cfg}: already built, skipping')
else:
    frames = {s: pd.concat([unify_ids(load('cicids2017', s)), unify_iot(load('ciciot2023', s))], ignore_index=True)
              for s in SPLITS}
    frames = {s: f.sample(frac=1, random_state=SEED).reset_index(drop=True) for s, f in frames.items()}
    X, params = fit_transform(frames, UNIFIED)
    save(cfg, X, frames, params, dict(mapping={k: list(v) for k, v in SAME.items()},
         corrected=dict(iat_mean='CICIDS Flow IAT Mean (us) | CIC-IoT2023 IAT x 1e6',
                        total_size='CICIDS Fwd + Bwd total length | CIC-IoT2023 Tot sum',
                        header_length='CICIDS Fwd Header Length / Total Fwd Packets, clipped to [0, 60] | CIC-IoT2023 Header_Length')))
    report[cfg] = dict(kept=len(UNIFIED), log_features=params['log_features'])
    # check: no unified feature may be (almost) constant for either dataset after scaling
    src = frames['train']['source'].to_numpy()
    chk = pd.DataFrame({d: X['train'][src == d].std(0) for d in ['CICIDS2017', 'CIC-IoT2023']}, index=UNIFIED)
    print('\nStd of scaled training values per dataset (should not be ~0 for both):')
    print(chk.round(4).to_string())
    del frames, X

In [ ]:
# Cell 4 — summary for Chapter 3 (feature tables)
for cfg in ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']:
    spec = json.load(open(FEAT + cfg + '/spec.json'))
    y = {s: np.load(FEAT + cfg + f'/y_{s}.npy') for s in SPLITS}
    print(f'\n================ {cfg} ================')
    print(f"features kept: {len(spec['features'])}"
          + (f" (of {spec['n_features_original']})" if 'n_features_original' in spec else ''))
    if 'dropped_zero_variance' in spec:
        print('dropped (zero variance):', spec['dropped_zero_variance'] or 'none')
        print('dropped (duplicate of):', spec['dropped_duplicates'] or 'none')
    print(f"log-transformed: {len(spec['log_features'])} -> {spec['log_features']}")
    counts = pd.DataFrame({s: pd.Series(v).value_counts() for s, v in y.items()}).reindex(range(len(CLASSES))).fillna(0).astype(int)
    counts.index = CLASSES
    print(counts.to_string())